<a href="https://colab.research.google.com/github/S-Ananth7/Genai_agent_foundation/blob/main/Simple_Conversational_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Important Required Libraries**

In [25]:
!pip install -q langchain langchain_experimental langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.3 MB/s eta 0:00:00


**Load the API key securely from Google Colab userdata and initialize the language model before using it in the agent.**

In [27]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [28]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


### **Sample Code From Official Documentation **

In [33]:
from langchain.agents import create_agent
import os

def get_weather(city:str) -> str:
  """Get weather for a given city"""
  return f"It's always sunny in {city}!"

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {
        "messages": [{"role": "user", "content":"What's the weather in Montgomery"}]
    }
)

print(result["messages"][-1].content)
print(result["messages"][-1].content_blocks)


It's always sunny in Montgomery!
[{'type': 'text', 'text': "It's always sunny in Montgomery!"}]


In [44]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

**Create a simple in-memory store for chat histories**

In [38]:
store = {}

def get_chat_history(session_id: str):
  if session_id not in store:
    store[session_id] = ChatMessageHistory()
  return store[session_id]

**Create the Prompt template**

In [41]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human","{input}")
])

**Combine the Prompt and model into a runnable chain**

In [45]:
chain = prompt | llm

**Wrap the chain with message history**

In [47]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key="input",
    history_messages_key="history"
)

**Example usage**

In [48]:
from langchain_community import chat_message_histories
session_id = "user_123"

response1 = chain_with_history.invoke(
    {"input": "Hello! How are you?"},
    config={"configurable": {"session_id": session_id}}
)

print("AI:", response1.content)

response2 = chain_with_history.invoke(
    {"input": "What was my previous message?"},
    config={"configurable": {"session_id": session_id}}
)

print("AI:", response2.content)

AI: Hello! I'm functioning perfectly, as always. Thank you for asking!

How can I help you today?
AI: Your previous message was: "Hello! How are you?"


**Print the conversation history**

In [49]:
print("\nConversation History:")
for message in store[session_id].messages:
  print(f"{message.type}: {message.content}")


Conversation History:
human: Hello! How are you?
ai: Hello! I'm functioning perfectly, as always. Thank you for asking!

How can I help you today?
human: What was my previous message?
ai: Your previous message was: "Hello! How are you?"
